In [39]:
import torch
# print(torch.cuda.is_available())

# Tensors

In [40]:
int_tensor = torch.tensor([1, 2, 3])
print(int_tensor.dtype)
float_tensor = torch.tensor([1.0, 2.0, 3.0])
print(float_tensor.dtype)
float_tensor_2 = int_tensor.to(torch.float32)
print(float_tensor_2.dtype)

torch.int64
torch.float32
torch.float32


In [41]:
tensor_1d = torch.tensor([1, 2, 3, 4])
# Returns a view if the tensor is contiguous, or clones the data into a new, contiguous memory block.
tensor_2d = tensor_1d.reshape(2, 2) 
view_2d = tensor_1d.view(2, 2) # Original data must be continues.
print(tensor_2d)
print(view_2d)

tensor([[1, 2],
        [3, 4]])
tensor([[1, 2],
        [3, 4]])


In [42]:
print(tensor_2d.T)
print(tensor_2d @ tensor_2d.T) # Matrix multiplication

tensor([[1, 3],
        [2, 4]])
tensor([[ 5, 11],
        [11, 25]])


# Graphs

In [43]:
from torch.nn import functional as F
a = torch.tensor([1.0], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
x = torch.tensor([1.0], requires_grad=True)
y = a * x + b
s = torch.sigmoid(y)
loss = F.binary_cross_entropy(s, y)
print(y)
print(s)
print(loss)

tensor([1.], grad_fn=<AddBackward0>)
tensor([0.7311], grad_fn=<SigmoidBackward0>)
tensor(0.3133, grad_fn=<BinaryCrossEntropyBackward0>)


Pytorch builds a computational graph internally if one of its terminal nodes hes the requires_grad attribute set to True.

In [44]:
from torch.autograd import grad
grad_a = grad(loss, a, retain_graph=True) # by default, grads are destroyed after computations
print(grad_a)

(tensor([-1.2689]),)


Calling .backward on the loss computes the gradients of all the leaf nodes in the graph, which get stored via tensors' .grad attributes.

In [45]:
loss.backward()
print(a.grad)

tensor([-1.2689])


# Perceptron
To implement a NN, we subclass torch.nn.Module and implement the forward method.
The layers are defined in  __init__.

In [46]:
class MyNeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        
        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 32),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(32, 24),
            torch.nn.ReLU(),

            # Output layer
            torch.nn.Linear(24, num_outputs)
        )
    
    def forward(self, x):
        logits = self.layers(x)
        return logits

In [47]:
model = MyNeuralNetwork(num_inputs=2, num_outputs=2)
print(model)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {num_params}")

MyNeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=24, bias=True)
    (3): ReLU()
    (4): Linear(in_features=24, out_features=2, bias=True)
  )
)
Number of trainable parameters: 938


In [48]:
# Feedforward - multimlication of input with weights, addition of bias, and application of activation function.
print(model.layers[0].weight)
print(model.layers[0].bias)
print(model.layers[0].weight.shape)
print(model.layers[0].bias.shape)

Parameter containing:
tensor([[ 0.2134, -0.4366],
        [ 0.3564, -0.6003],
        [ 0.4102, -0.4962],
        [-0.4645, -0.2537],
        [-0.1543, -0.6391],
        [ 0.5709,  0.0305],
        [ 0.1860, -0.2631],
        [-0.0678, -0.3377],
        [-0.3625, -0.4821],
        [ 0.4705, -0.4073],
        [-0.1834, -0.0840],
        [ 0.5584, -0.0793],
        [ 0.6535,  0.4897],
        [ 0.3715,  0.4320],
        [ 0.1272,  0.5131],
        [-0.6803, -0.0451],
        [-0.4969,  0.5543],
        [-0.6826, -0.3404],
        [ 0.2456,  0.4351],
        [ 0.4299, -0.4357],
        [-0.6235, -0.1554],
        [-0.6928,  0.4854],
        [-0.3721,  0.1568],
        [-0.0908, -0.6989],
        [-0.6299, -0.3871],
        [ 0.3641,  0.4585],
        [ 0.5074, -0.2103],
        [ 0.4414,  0.1698],
        [ 0.0592, -0.6957],
        [-0.5437, -0.2613],
        [ 0.6678, -0.3631],
        [-0.6720,  0.6802]], requires_grad=True)
Parameter containing:
tensor([-0.3148,  0.1725,  0.5067, -0.6

In [49]:
torch.manual_seed(101)  # For reproducibility
x = torch.rand(1, 2)  # Random input tensor with shape (1, 2)
# For inference, there's no need to keep track of the gradients.
with torch.no_grad():
    y = model(x)
    out = torch.softmax(y, dim=1)
print(out)

tensor([[0.5794, 0.4206]])


# Datasets & Data Loaders

In [50]:
x_train = torch.randint(0, 2, (2, 50)).to(torch.float32)
x_test = torch.randint(0, 2, (2, 5)).to(torch.float32)
y_train = torch.logical_xor(x_train[0], x_train[1]).long()
y_test = torch.logical_xor(x_test[0], x_test[1]).long()
print(x_test)
print(y_test)


tensor([[0., 1., 0., 1., 0.],
        [1., 1., 1., 0., 0.]])
tensor([1, 0, 1, 1, 0])


In [51]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, x, y):
        self.features = x
        self.labels = y

    def __getitem__(self, index):
        x_sample = self.features[:, index]
        y_sample = self.labels[index]
        return x_sample, y_sample

    def __len__(self):
        return len(self.labels)

train_dataset = MyDataset(x_train, y_train)
test_dataset = MyDataset(x_test, y_test)

In [85]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=10, 
    shuffle=True, 
    num_workers=0, # data loading is done in the main process, not in a separate worker
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=1, 
    shuffle=False, 
    num_workers=0, # 4 is usually optimal
    drop_last=True
)

for inx, (x_batch, y_batch) in enumerate(test_loader):
    print(f"Batch {inx+1}:")
    print(f"samples: {x_batch}")
    print(f"labels: {y_batch}")

Batch 1:
samples: tensor([[0., 1.]])
labels: tensor([1])
Batch 2:
samples: tensor([[1., 1.]])
labels: tensor([0])
Batch 3:
samples: tensor([[0., 1.]])
labels: tensor([1])
Batch 4:
samples: tensor([[1., 0.]])
labels: tensor([1])
Batch 5:
samples: tensor([[0., 0.]])
labels: tensor([0])


# Training

In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.05
)

num_epochs = 3
for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    for inx, (features, labels) in enumerate(train_loader):
        # Forward pass
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()  # Clear gradients from the previous step
        loss.backward()  # Backpropagation to compute gradients
        optimizer.step()  # Update model parameters using the computed gradients

        print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{inx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

Epoch [1/3], Batch [1/5], Loss: 0.0032
Epoch [1/3], Batch [2/5], Loss: 0.0033
Epoch [1/3], Batch [3/5], Loss: 0.0035
Epoch [1/3], Batch [4/5], Loss: 0.0035
Epoch [1/3], Batch [5/5], Loss: 0.0040
Epoch [2/3], Batch [1/5], Loss: 0.0038
Epoch [2/3], Batch [2/5], Loss: 0.0028
Epoch [2/3], Batch [3/5], Loss: 0.0033
Epoch [2/3], Batch [4/5], Loss: 0.0030
Epoch [2/3], Batch [5/5], Loss: 0.0043
Epoch [3/3], Batch [1/5], Loss: 0.0019
Epoch [3/3], Batch [2/5], Loss: 0.0039
Epoch [3/3], Batch [3/5], Loss: 0.0039
Epoch [3/3], Batch [4/5], Loss: 0.0047
Epoch [3/3], Batch [5/5], Loss: 0.0026


# Evaluation

In [100]:
model.eval() # Set the model to evaluation mode
with torch.no_grad():
    logits = model(x_test.T)
    predictions = torch.argmax(logits, dim=1)

correct = predictions == y_test
print(correct)

accuracy = correct.float().mean()
print(f"Accuracy: {accuracy:.4f}")


tensor([True, True, True, True, True])
Accuracy: 1.0000


# Saving & Loading

In [ ]:
torch.save(model.state_dict(), "my_model.pth")

loaded_model = MyNeuralNetwork(num_inputs=2, num_outputs=2)
loaded_model.load_state_dict(torch.load("my_model.pth"))

MyNeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=24, bias=True)
    (3): ReLU()
    (4): Linear(in_features=24, out_features=2, bias=True)
  )
)
